### RAG PIPELINE - DATA INGESTION TO VECTOR DB PIPELINE

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\PEM\AppData\Local\Temp\ipykernel_17732\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### READ ALL THE PDFS INSIDE THE DIRECTOY

def process_all_pdfs(pdf_directory):
    """ process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    """Find all pdf files recursively"""
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f'Found {len(pdf_files)} PDF Files to process')

    for pdf_file in pdf_files:
        print(f'\nProcessing : {pdf_file.name}')

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

    ## add source information to meta data

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'


            all_documents.extend(documents)
            print(f' Loaded {len(documents)} pages')

        except Exception as e:
            print(f'Error:{e}')
    
    print(f'\nTotal documents loaded:{len(all_documents)}')
    return all_documents

    """process all pdfs in the data directory"""

all_pdf_documents = process_all_pdfs("../data")
    

Found 4 PDF Files to process

Processing : early_marriage.pdf
 Loaded 1 pages

Processing : education_is_power.pdf
 Loaded 1 pages

Processing : Iqra Abid cover letter.pdf
 Loaded 1 pages

Processing : Iqra_Abid.pdf
 Loaded 1 pages

Total documents loaded:4


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-29T10:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-29T10:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\early_marriage.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'early_marriage.pdf', 'file_type': 'pdf'}, page_content='Early Marriage\nEarly marriage refers to marrying before adulthood, often before the age of 18. It can limit\neducation, reduce career opportunities, and increase health risks, especially for girls. Young\ncouples may not be emotionally or financially prepared for family responsibilities. Communities and\ngovernments can reduce early marriage by promoting education, enforcing minimum-age laws, and\nraising awareness about its long-term effects. Supporting young people to complete their education\nhelps them make informed life de

In [4]:
###text splitting get into chunks

def split_documents(documents, chunk_size =1000 , chunk_overlap = 500):
    """split documents into smaller chunks for better rag performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f'split {len(documents)} documents into {len(split_docs)} chunks')

    #show example of a chunk
    if split_docs:
        print("\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f'Metadata:{split_docs[0].metadata}')

    return split_docs

    

In [5]:
chunk = split_documents(all_pdf_documents)

split 4 documents into 11 chunks

Example Chunk:
Content: Early Marriage
Early marriage refers to marrying before adulthood, often before the age of 18. It can limit
education, reduce career opportunities, and increase health risks, especially for girls. You...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-29T10:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-29T10:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\early_marriage.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'early_marriage.pdf', 'file_type': 'pdf'}


In [6]:
chunk

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-29T10:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-29T10:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\early_marriage.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'early_marriage.pdf', 'file_type': 'pdf'}, page_content='Early Marriage\nEarly marriage refers to marrying before adulthood, often before the age of 18. It can limit\neducation, reduce career opportunities, and increase health risks, especially for girls. Young\ncouples may not be emotionally or financially prepared for family responsibilities. Communities and\ngovernments can reduce early marriage by promoting education, enforcing minimum-age laws, and\nraising awareness about its long-term effects. Supporting young people to complete their education\nhelps them make informed life de

### embedding and vector store db


In [7]:
!pip install sentence-transformers


[notice] A new release of pip available: 22.3.1 -> 26.2
[notice] To update, run: C:\Users\PEM\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [8]:
import chromadb
print(chromadb.__version__)

1.5.9


In [9]:
import numpy as numpy
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid  #every record that insert into vector db should have an id
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using sentence transfomaers
    """
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):  #this is the model present in hugging face that conevrt text into vectors
        """" initialize the embedding manager
        args:
        model_name : hugging face model name  for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """load the sentence transformer model"""
        try:
            print(f'loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully.Embedding Dimension : {self.model.get_embedding_dimension()}')

        except Exception as e:
            print(f'Error loading model {self.model_name}: {e}')
            raise

    def generate_embeddings(self, texts:List[str])-> np.ndarray:
        """generate embeddings for a list of texts 

        args:
               texts: list of text string to embed
        Returns:
              numpy arrays of embeddings with shape (len(texts), embedding_dim)"""

        if not self.model:
            raise ValueError("model not loaded")
        print(f'generating embedings for {len(texts)} texts...')
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f'Generated embeddings with shape: {embeddings.shape}')
        return embeddings


### initialize the embedding manager
Embedding_Manager = EmbeddingManager()
Embedding_Manager

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2526.64it/s]


Model loaded successfully.Embedding Dimension : 384


### VECTOR STORE


In [11]:
import os
import chromadb
class Vector_Store:
    """manages document embeddings in a chromadb vector store"""

    def __init__(self , collection_name: str = "pdf_documents", persist_directory :str = "../data/vector_store"):

        """initialize the vector store
        Args:
        collection_name : Name of chromadb collection
        persist_directory: directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        """initialize chromadb client and collection"""
        try:
            #create persistant chromadb client

            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient (path=self.persist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description":"pdf document embeddings for RAG"}
            )

            print(f'Vector store initialized.Collection: {self.collection_name}')
            print(f'existing documents in collection: {self.collection.count()}')

        except Exception as e:
            print(f'error initializing vector store: {e}')
            raise 

    def add_documents(self,documents: List[Any], embeddings : np.ndarray):
        """add documents and their embeddings to the vector store
        args:
        documents: list of langchain documents
        embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vectore store..")


        #prepare data for chroma db
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate (zip(documents,embeddings)):
            #generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #documents
            documents_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(embedding.tolist())

        try:
             self.collection.add(
                 ids = ids,
                 embeddings = embeddings_list,
                 metadatas=metadatas,
                 documents = documents_text

             )
             print(f'Successfully added {len(documents)} documents to vector store')
             print(f'Total documents in collection: {self.collection.count()}')

        except Exception as e:
            print(f'Error adding documents to vector store: {e}')
            raise

vectorstore = Vector_Store()
vectorstore
        



Vector store initialized.Collection: pdf_documents
existing documents in collection: 46


In [12]:
chunk

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-29T10:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-29T10:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\early_marriage.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'early_marriage.pdf', 'file_type': 'pdf'}, page_content='Early Marriage\nEarly marriage refers to marrying before adulthood, often before the age of 18. It can limit\neducation, reduce career opportunities, and increase health risks, especially for girls. Young\ncouples may not be emotionally or financially prepared for family responsibilities. Communities and\ngovernments can reduce early marriage by promoting education, enforcing minimum-age laws, and\nraising awareness about its long-term effects. Supporting young people to complete their education\nhelps them make informed life de

In [13]:
#Convert the text to embeddings

textss = [doc.page_content for doc in chunk]
textss

['Early Marriage\nEarly marriage refers to marrying before adulthood, often before the age of 18. It can limit\neducation, reduce career opportunities, and increase health risks, especially for girls. Young\ncouples may not be emotionally or financially prepared for family responsibilities. Communities and\ngovernments can reduce early marriage by promoting education, enforcing minimum-age laws, and\nraising awareness about its long-term effects. Supporting young people to complete their education\nhelps them make informed life decisions and build a better future.',
 'Education Is Power\nEducation gives people knowledge, skills, and confidence. It helps individuals make informed\ndecisions, solve problems, and contribute positively to society. Education opens doors to better\njobs, improved health, and greater opportunities. An educated society is more innovative,\nproductive, and peaceful. By encouraging lifelong learning and equal access to education,\ncommunities can reduce poverty 

In [14]:
#generate the embeddings
embedding_manager = EmbeddingManager()

embeddings = embedding_manager.generate_embeddings(textss)

#store in the vector db

vectorstore = Vector_Store()

vectorstore.add_documents(chunk, embeddings)


loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2095.80it/s]


Model loaded successfully.Embedding Dimension : 384
generating embedings for 11 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Generated embeddings with shape: (11, 384)
Vector store initialized.Collection: pdf_documents
existing documents in collection: 46
Adding 11 documents to vectore store..
Successfully added 11 documents to vector store
Total documents in collection: 57


### Retrieval pipeline from vector store


In [31]:
class RAGRetriever:
    """handles query-based retrieval from the vector store
"""
    def __init__(self, vectorstore: Vector_Store, embedding_manager: EmbeddingManager):
        """initialize the retriever 
        Args:
        vectorstore: vector store containing document embeddings
        embedding_manager: manager for generating query embeddings
        """

        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str , top_k:int=5, score_threshold: float = 0.0) -> List[dict[str,Any]]:
        
        """retrieve relevant documents for a query
        
        Args:
         query: the search query 
         top_k: number of top results to return
         score_threshold: minimum similarity score threshold
         
        Returns:
         List of dictionaries conatining retrieved documents and metadata"""


        print(f"Retreiving documents for query: '{query}'")
        print(f'Top_k:{top_k}, Score_threshold:{score_threshold}')

        #generate query embedding 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results= top_k
            )

            #process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id,document,metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance to similarity score (chromadb uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f'Retrieved {len(retrieved_docs)}documents (after filtering)')

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f'Error during retrieval: {e}')
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever












In [33]:
rag_retriever.retrieve("what is early marriage")

Retreiving documents for query: 'what is early marriage'
Top_k:5, Score_threshold:0.0
generating embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2documents (after filtering)


[{'id': 'doc_1156cc8c_0',
  'content': 'Early Marriage\nEarly marriage refers to marrying before adulthood, often before the age of 18. It can limit\neducation, reduce career opportunities, and increase health risks, especially for girls. Young\ncouples may not be emotionally or financially prepared for family responsibilities. Communities and\ngovernments can reduce early marriage by promoting education, enforcing minimum-age laws, and\nraising awareness about its long-term effects. Supporting young people to complete their education\nhelps them make informed life decisions and build a better future.',
  'metadata': {'author': '(anonymous)',
   'trapped': '/False',
   'doc_index': 0,
   'moddate': '2026-07-29T10:56:47+00:00',
   'keywords': '',
   'total_pages': 1,
   'file_type': 'pdf',
   'creationdate': '2026-07-29T10:56:47+00:00',
   'page': 0,
   'source': '..\\data\\pdf\\early_marriage.pdf',
   'producer': 'ReportLab PDF Library - (opensource)',
   'title': '(anonymous)',
   'su

In [34]:
rag_retriever.retrieve("Leadership")

Retreiving documents for query: 'Leadership'
Top_k:5, Score_threshold:0.0
generating embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2documents (after filtering)


[{'id': 'doc_df3feaec_12',
  'content': '• Contributed to planning and executing student-led activities through effective teamwork and creative leadership.',
  'metadata': {'keywords': '',
   'creationdate': '2026-07-13T12:34:01+00:00',
   'source': '..\\data\\pdf\\Iqra_Abid.pdf',
   'subject': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1',
   'moddate': '2026-07-13T12:34:01+00:00',
   'producer': 'pdfTeX-1.40.27',
   'source_file': 'Iqra_Abid.pdf',
   'content_length': 114,
   'trapped': '/False',
   'title': '',
   'page': 0,
   'file_type': 'pdf',
   'author': '',
   'creator': 'LaTeX with hyperref',
   'doc_index': 12,
   'page_label': '1',
   'total_pages': 1},
  'similarity_score': 0.10133308172225952,
  'distance': 0.8986669182777405,
  'rank': 1},
 {'id': 'doc_98081563_12',
  'content': '• Contributed to planning and executing student-led activities through effective teamwork and creative leadership.',
  'meta

In [36]:
import sys
print(sys.executable)
print(sys.version)

c:\RAG\.venv\Scripts\python.exe
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


In [37]:
import subprocess
subprocess.run(["python", "-m", "pip", "show", "langchain-groq"])

CompletedProcess(args=['python', '-m', 'pip', 'show', 'langchain-groq'], returncode=1)

In [38]:
%pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [21]:
import sys
!{sys.executable} -m pip install langchain-groq

c:\RAG\.venv\Scripts\python.exe: No module named pip


### Integration vectordb conetxt pipeline with LLM output

In [ ]:
#simple RAG pipeline with groq llm
!pip install langchain-groq
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq LLM (set your groq api key in environment)
groq_api_key = ""

llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature = 0.1,max_tokens=1024)

### Simple RAG function : retreive context + generate context
def rag_simple(query,retriever,llm,top_k=3):
    ##retrieve the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context: 
        return "No relevant context found to answer the question"

    ###Generate the answer using GROQ LLM
    prompt= f"""Use the following context to answer the question concisely.
        Context : {context}
        Question : {query}
        Answer : 
"""
    response = llm.invoke([prompt.format(context=context,query= query)])
    return response.content


In [ ]:
#simple RAG pipeline with groq llm
!pip install langchain-groq
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq LLM (set your groq api key in environment)
GROQ_API_KEY=your_new_key_here

llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature = 0.1,max_tokens=1024)

### Simple RAG function : retreive context + generate context
def rag_simple(query,retriever,llm,top_k=3):
    ##retrieve the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context: 
        return "No relevant context found to answer the question"

    ###Generate the answer using GROQ LLM
    prompt= f"""Use the following context to answer the question concisely.
        Context : {context}
        Question : {query}
        Answer : 
"""
    response = llm.invoke([prompt.format(context=context,query= query)])
    return response.content


In [51]:
answer = rag_simple("What is early marriage?",rag_retriever,llm)
print(answer)

Retreiving documents for query: 'What is early marriage?'
Top_k:3, Score_threshold:0.0
generating embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2documents (after filtering)


Early marriage refers to marrying before adulthood, often before the age of 18.


Enhanced RAG pipeline features

In [54]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    context = "\n\n".join(doc["content"] for doc in results)

    sources = [
        {
            "source": doc["metadata"].get(
                "sourcefile",
                doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "..."
        }
        for doc in results
    ]

    confidence = max(doc["similarity_score"] for doc in results)

    prompt = f"""
Use the following context to answer the question precisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context

    return output

In [56]:
# Example usage
result = rag_advanced(
    query="Alphabet recognition system",
    retriever=rag_retriever,
    llm=llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:")
print(result["answer"])

print("\nConfidence:")
print(result["confidence"])

print("\nSources:")
for source in result["sources"]:
    print(source)

print("\nContext Preview:")
print(result["context"][:300])

Retreiving documents for query: 'Alphabet recognition system'
Top_k:3, Score_threshold:0.1
generating embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2documents (after filtering)


Answer:
Alphabet Recognition System: 
Iqra Abid built a custom dataset of alphabet images and trained a machine learning model for real-time character recognition as part of her academic project.

Confidence:
0.29273319244384766

Sources:
{'source': '..\\data\\pdf\\Iqra_Abid.pdf', 'page': 0, 'score': 0.29273319244384766, 'preview': 'Iqra Abid\n♂¶ap-¶arker-altKarachi, Pakistan\n♂phone03272933893 /envel⌢peiqra65952@gmail.com /linkedinlinkedin.com/in/IqraAbid /githubgithub.com/Iqra-Abid\nEducation\nNED University 2023-2027\nBE Computer Systems Engineering CGPA : 3.34\nAcademic Projects\nAlphabet Recognition System\nMachine Learning Projec...'}
{'source': '..\\data\\pdf\\Iqra_Abid.pdf', 'page': 0, 'score': 0.29273319244384766, 'preview': 'Iqra Abid\n♂¶ap-¶arker-altKarachi, Pakistan\n♂phone03272933893 /envel⌢peiqra65952@gmail.com /linkedinlinkedin.com/in/IqraAbid /githubgithub.com/Iqra-Abid\nEducation\nNED University 2023-2027\nBE Computer Systems Engineering CGPA : 3.34\nAcademic Projects\

In [58]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Alphabet recognition system", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retreiving documents for query: 'Alphabet recognition system'
Top_k:3, Score_threshold:0.1
generating embedings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Iqra Abid
♂¶ap-¶arker-altKarachi, Pakistan
♂phone03272933893 /envel⌢peiqra65952@gmail.com /linkedinlinkedin.com/in/IqraAbid /githubgithub.com/Iqra-Abid
Education
NED Unive

rsity 2023-2027
BE Computer Systems Engineering CGPA : 3.34
Academic Projects
Alphabet Recognition System
Machine Learning Project
• Built a custom dataset of alphabet images and trained a machine learning model for real-time character recognition.

Iqra Abid
♂¶ap-¶arker-altKarachi, Pakistan
♂phone03272933893 /envel⌢peiqra65952@gmail.com /linkedinlinkedin.com/in/IqraAbid /githubgithub.com/Iqra-Abid
Education
NED University 2023-2027
BE Computer Systems Engineering CGPA : 3.34
Academic Projects
Alphabet Recognition System
Machine Learning Project
• Built a custom dataset of alphabet images and trained a machine learning model for real-time character recognition.

Question: Alphabet recognition system

Answer:

Final Answer: Iqra Abid built an Alphabet Recognition System as part of her academic projects, which involves building a custom dataset of alphabet images and training a machine learning model for real-time character recognition.

Citations:
[1] Iqra_Abid.pdf (page 0)
[2] Iqra_Abi